<a href="https://colab.research.google.com/github/Cristiano-Corsi-Unipi/MIRCV_RAG_project/blob/main/MIRCV_RAG_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compliance Checker: Automated Decision Tree Verification

**Authors:** Lorenzo Ceccanti, Cristiano Corsi, Rojan Shrestha

*Master of Science in Artificial Intelligence and Data Engineering*

---

## Introduction

In an era where every device and system is connected to the internet, cybersecurity compliance has become a critical business imperative. Yet compliance verification processes remain a significant operational bottleneck, with organizations spending countless hours manually reviewing product documentation against regulatory frameworks — a labor-intensive process prone to human error and inconsistency.

This project aims to build an **automated tool for compliance checking** that can help assessors transform weeks of manual review into minutes of automated analysis, enabling organizations to dramatically accelerate time-to-market while reducing compliance-related costs. By leveraging **Retrieval-Augmented Generation (RAG)** combined with decision tree evaluation, the system analyzes product documentation against industry-standard cybersecurity frameworks, automatically determining compliance status across multiple regulatory dimensions including security controls, authentication mechanisms, cryptography standards, and update management protocols.

### The Challenge

Traditional compliance verification requires teams of compliance specialists to manually cross-reference technical specifications with regulatory requirements — a process that is:
- **Time-intensive**: Taking weeks to complete comprehensive reviews
- **Error-prone**: Subject to human oversight and inconsistent interpretations
- **Costly**: Requiring significant allocation of specialized human resources
- **Non-scalable**: Becoming exponentially complex when dealing with multiple frameworks simultaneously

### Our Goal

We wanted to build an automated tool that could help assessors in compliance checking. To do so, we addressed key challenges by:

1. **PDF Document Preprocessing**: Using `Unstructured` library with a **Vision LLM** for handling complex elements like images and tables
2. **RAG System Construction**: Creating a vector-based retrieval system using sentence transformers and ChromaDB for intelligent context retrieval
3. **Decision Tree Parsing**: Parsing PlantUML-formatted decision trees that encode compliance requirements.
4. **Automated Evaluation**: Using a **LLM** to evaluate each requirement with retrieved context, delivering clear verdicts with detailed justifications.
5. **Tree Navigation**: Automatically navigating decision trees based on LLM responses.

### Use Case: European RED Directive and EN 18031-1 Standard

To demonstrate the system's capabilities, we focus on a concrete regulatory scenario: compliance verification for IoT devices under the **European Radio Equipment Directive (RED)** and the harmonized standard **EN 18031-1**. This standard defines cybersecurity requirements for radio equipment, covering critical aspects such as secure communication protocols, authentication mechanisms, cryptographic implementations, and software update management.

Our test case analyzes the **CC3200 SimpleLink™ Wi-Fi® and Internet of Things Solution with MCU LaunchPad™ Hardware** from Texas Instruments — a representative IoT device widely used in connected applications. The system processes the [device's technical documentation](https://www.ti.com/lit/ug/swru372c/swru372c.pdf) and evaluates it against decision trees provided by the [EN 18031-1 standard](https://drive.google.com/file/d/1lxRBuPoY-8km09MEKT_bnFrtSh0Fnpwq/view?usp=sharing), automatically determining compliance with RED requirements.

This real-world scenario demonstrates how the automated approach can be applied to actual regulatory frameworks and commercial products.

### LLM Backend

The system supports two backends:
- **llama.cpp (local)**: For local development with your own GPU/CPU server
- **OpenRouter API (remote)**: For cloud execution (e.g., Google Colab) using the free tier

---

## 1. Setup and Installation

### Installing Dependencies

In [ ]:
import platform
import sys

In [ ]:
URL_PRODUCT_DESCRIPTION = "https://drive.google.com/file/d/171VDoWwySebZVCVTd6tKuCpxI0NeHVWh/view?usp=sharing"
URL_DECISION_TREES = "https://drive.google.com/file/d/19UKLuODBeMI268GKV5P5p3CkiIV6rmE5/view?usp=sharing"
URL_MODELS = "https://drive.google.com/file/d/1VsMSPdUUW0ADF-gCm5_wiD8rs50skGY7/view?usp=sharing"

# Detect operating system
IS_WINDOWS = platform.system() == "Windows"
IS_MACOS_ARM = "macos" in platform.platform().lower() and "arm64" in platform.platform().lower()
IS_COLAB = "google.colab" in sys.modules

In [ ]:
import_cmd = "pip install unstructured[pdf] pytesseract pdf2image sentence-transformers huggingface_hub[hf_xet] chromadb PyMuPDF tqdm pandas numpy python-dotenv pillow gdown"
if IS_MACOS_ARM:
    arr_cmds = import_cmd.split()
    cmd_begin = arr_cmds[0:2]
    
    # At the beginning contains pip install
    # Then we add quotes for all the packages encountered (the MacOS terminal confuses the [] with a pattern to match, instead we intend a pip package)
    cmd_end = cmd_begin
    
    for package in arr_cmds[2:]:
        package = "'" + package + "'"
        cmd_end.append(package)
    import_cmd = " ".join(cmd_end)
print("Executing command: ", import_cmd)
!{import_cmd}

### Installing llama-cpp-python with CUDA Support

We install `llama-cpp-python` with CUDA support.

The installation uses the official CUDA wheels from the `llama-cpp-python` repository. Make sure you have:
- Python 3.10, 3.11, or 3.12
- CUDA 12.1-12.5 compatible drivers (can be checked with `nvidia-smi`)

In [ ]:
import platform

# Check OS type
current_os = platform.system()

if IS_COLAB:
    # Installation with prebuilt CUDA wheels for COLAB
    !{sys.executable} -m pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 --verbose   
    print("llama-cpp-python installed with CUDA support (COLAB)")
elif current_os == "Windows":
    # Windows: use prebuilt CUDA wheels
    CUDA_WHL = "cu125"  # Set the CUDA version of the system
    !{sys.executable} -m pip install https://github.com/dougeeai/llama-cpp-python-wheels/releases/download/v0.3.16-cuda13.0-sm86-py312/llama_cpp_python-0.3.16+cuda13.0.sm86.ampere-cp312-cp312-win_amd64.whl
    print(f"llama-cpp-python installed with CUDA support ({CUDA_WHL})")
elif current_os == "Darwin":  # macOS
    # macOS: build with Metal support for Apple Silicon GPU acceleration
    !CMAKE_ARGS="-DGGML_METAL=on" {sys.executable} -m pip install llama-cpp-python
    print("llama-cpp-python installed with Metal support (Apple GPU)")
else:
    # Linux: build from source with CUDA support
    !CMAKE_ARGS="-DGGML_CUDA=on" {sys.executable} -m pip install llama-cpp-python
    print("llama-cpp-python installed with CUDA support")

### Importing Libraries

In [ ]:
import os
import gc
import json
import base64
import random
import hashlib
import pickle
import sys
import platform
import gdown
import tarfile
import zipfile
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import fitz  # PyMuPDF
import numpy as np
from PIL import Image as PILImage
import io

import torch
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Llava15ChatHandler

import chromadb
from chromadb.config import Settings

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from unstructured.documents.elements import Image, Table, Text, Title, NarrativeText, ListItem, FigureCaption

from dotenv import load_dotenv

import subprocess
import shutil

# Load environment variables
load_dotenv()

Unstructured - the library used to parse and partition PDFs - requires **Tesseract OCR** and **Poppler** in order to work with the `hi_res` strategy. 

#### Windows Installation Options

Both tools can be configured in three ways (checked in this order):

1. **`.env` file** - Set `TESSERACT_PATH` and/or `POPPLER_PATH`
2. **System PATH** - Install globally and add to PATH
3. **Local `./tools/` directory** - Extract portable versions

> **Note**: Windows users need to manually install both Tesseract and Poppler in order to make the system work.
> - Tesseract must be installed globally and added to path or the path can be set in the .env file
> - Poppler can be installed in `./tools/poppler`

In [ ]:
# Detect operating system
IS_WINDOWS = platform.system() == "Windows"
IS_COLAB = "google.colab" in sys.modules

# Local installation directory for external tools (keeps project clean)
LOCAL_TOOLS_DIR = Path("./tools")
LOCAL_TOOLS_DIR.mkdir(exist_ok=True)

print(f"Operating System: {platform.system()}")
print(f"Running on Colab: {IS_COLAB}")
print(f"Local tools directory: {LOCAL_TOOLS_DIR.absolute()}")

# =============================================================================
# HELPER FUNCTION: Find external tool
# =============================================================================
def find_tool(tool_name: str, executable: str, local_paths: list[Path] = None) -> str | None:
    """
    Find an external tool with priority: 1) .env  2) System PATH  3) Local paths
    Returns the directory containing the executable, or None if not found.
    """
    # 1) Check .env
    env_path = os.getenv(f"{tool_name.upper()}_PATH", None)
    if env_path and Path(env_path).exists():
        print(f"✓ {tool_name} found (.env): {env_path}")
        return env_path
    
    # 2) Check system PATH
    system_exe = shutil.which(executable)
    if system_exe:
        tool_dir = str(Path(system_exe).parent)
        print(f"✓ {tool_name} found (PATH): {tool_dir}")
        return tool_dir
    
    # 3) Check local paths (Windows only)
    if IS_WINDOWS and local_paths:
        for local_path in local_paths:
            if local_path.exists() and (local_path / f"{executable}.exe").exists():
                tool_dir = str(local_path)
                print(f"✓ {tool_name} found (local): {tool_dir}")
                return tool_dir
    
    return None

# =============================================================================
# FIND/INSTALL TESSERACT OCR
# =============================================================================
TESSERACT_PATH = find_tool(
    "Tesseract",
    "tesseract",
    [LOCAL_TOOLS_DIR / "tesseract"]  # Possible local installation
)

if not TESSERACT_PATH:
    if IS_WINDOWS:
        print("⚠ Tesseract not found. Options:")
        print("  1. Install globally and add to PATH")
        print("  2. Set TESSERACT_PATH in .env file")
    elif IS_MACOS_ARM:
        print("Installing Tesseract OCR (via MacPorts) ...")
        # Checking if MacPorts is installed. If is not installed, the output is returned to stderr stream
        cmd = "port version"
        output = subprocess.run(cmd, capture_output=True, text=True, shell=True)
        if output.stderr != '':
            # If the stderr stream is not empty, we need to install MacPorts
            print("[ERR](STDERR): ", output.stderr.split("\n"))
            print("MacPort is required to be installed on your MacOS machine")
            print("Follow the instruction here: https://www.macports.org/install.php")
        else:
            # MacPorts installation exists..
            print("MacPorts found", output.stdout.split("\n")[:-1])
            # Installing tesseract
            # Asking sudo password
            # sudo expects an environment variable called SUDO_ASKPASS which contains
            # the path to the script opening the window for typing the password
            cmd = "SUDO_ASKPASS=askpass.sh sudo -A port install tesseract"
            subprocess.run(cmd, shell=True, check=True)
            TESSERACT_PATH = str(Path(shutil.which("tesseract")).parent)
            cmd = "SUDO_ASKPASS=askpass.sh sudo -A port install tesseract-eng"
            subprocess.run(cmd, shell=True, check=True)
            cmd = "SUDO_ASKPASS=askpass.sh sudo -A port install tesseract-osd"
            subprocess.run(cmd, shell=True, check=True)
            print(f"Tesseract installed: {TESSERACT_PATH}")
    else:
        # Linux/Colab: auto-install
        print("Installing Tesseract OCR...")
        # Detect Linux distribution
        if Path("/etc/arch-release").exists():
            # Arch Linux
            subprocess.run(["sudo", "pacman", "-Sy", "--noconfirm", "tesseract", "tesseract-data-eng"], check=True)
        else:
            # Debian/Ubuntu/Colab
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr"], check=True)
        TESSERACT_PATH = str(Path(shutil.which("tesseract")).parent)
        print(f"Tesseract installed: {TESSERACT_PATH}")

# =============================================================================
# FIND/INSTALL POPPLER (for pdf2image)
# =============================================================================
POPPLER_PATH = find_tool(
    "Poppler",
    "pdfinfo",
    [
        LOCAL_TOOLS_DIR / "poppler" / "Library" / "bin",  # conda-style layout
        LOCAL_TOOLS_DIR / "poppler" / "bin",              # standard layout
    ]
)

if not POPPLER_PATH:
    if IS_WINDOWS:
        print("⚠ Poppler not found. Options:")
        print("  1. Install globally and add to PATH")
        print("  2. Set POPPLER_PATH in .env file")
        print("  3. Extract to ./tools/poppler/")
        print("  Download: https://github.com/oschwartz10612/poppler-windows/releases")
    elif IS_MACOS_ARM:
        print("Checking to have the a 16.x Xcode tools version installed...")
        cmd = "xcode-select --install"
        output = subprocess.run(cmd, capture_output=True, text=True, shell=True)
        if output.stderr != '':
            # xcode tools already installed, checking the version installed
            # on MacOS to install Poppler a 16.x Xcode tools version installed is required
            cmd = "pkgutil --pkg-info=com.apple.pkg.CLTools_Executables"
            output = subprocess.run(cmd, capture_output=True, text=True, shell=True)
            # We're interested in the version, which is the second line
            # the second line is organized as follows
            # version: <versionNumber>
            version = output.stdout.split("\n")[1].split(":")[1].strip()
            if not version.startswith("16"):
                
                # Retrieving the list of compatible Xcode tools version to be installed in the machine
                print("Update of Xcode tools required. Updating...")
                cmd = "softwareupdate --list"
                output = subprocess.run(cmd, capture_output=True, text=True, shell=True)
                sw_upd_out = output.stdout.split("\n")
                for label in sw_upd_out:
                    if label.startswith("* Label: Command Line Tools for Xcode-16."):
                        # We've found a compatible version for Xcode tools to be installed
                        # Removing the trailing * character, which is undesired for the next command
                        label = label.strip("* Label:")
                        break
                
                cmd = f'SUDO_ASKPASS=askpass.sh sudo -A softwareupdate --install "{label}"'
                print("[EXECUTING COMMAND]: ", cmd)
                subprocess.run(cmd, shell=True, check=True)
                
        print("All set. Installing Poppler utilities (via MacPorts)...")
        cmd = "SUDO_ASKPASS=askpass.sh sudo -A port install poppler"
        subprocess.run(cmd, shell=True, check=True)
        POPPLER_PATH = str(Path(shutil.which("pdfinfo")).parent)
        print(f"Poppler installed: {POPPLER_PATH}")
    else:
        # Linux/Colab: auto-install
        print("Installing Poppler utilities...")
        # Detect Linux distribution
        if Path("/etc/arch-release").exists():
            # Arch Linux
            subprocess.run(["sudo", "pacman", "-Sy", "--noconfirm", "poppler"], check=True)
        else:
            # Debian/Ubuntu/Colab
            subprocess.run(["apt-get", "install", "-y", "-qq", "poppler-utils"], check=True)
        POPPLER_PATH = str(Path(shutil.which("pdfinfo")).parent)
        print(f"Poppler installed: {POPPLER_PATH}")

# =============================================================================
# INSTALL LIBMAGIC (Linux only, for file type detection)
# =============================================================================
if not IS_WINDOWS:
    if not shutil.which("file"):
        print("Installing libmagic...")
        # Detect Linux distribution
        if Path("/etc/arch-release").exists():
            # Arch Linux - file command is in core/file package
            subprocess.run(["sudo", "pacman", "-Sy", "--noconfirm", "file"], check=True)
        else:
            # Debian/Ubuntu/Colab
            subprocess.run(["apt-get", "install", "-y", "-qq", "libmagic1"], check=True)
        print("libmagic installed")
    else:
        print("libmagic found")

# =============================================================================
# CONFIGURE TOOLS AND ADD TO PROCESS PATH
# =============================================================================
def add_to_process_path(tool_path: str, tool_name: str):
    """Add tool directory to process PATH if not already present."""
    if tool_path:
        current_path = os.environ.get("PATH", "")
        if tool_path not in current_path:
            os.environ["PATH"] = tool_path + os.pathsep + current_path
            print(f"{tool_name} added to process PATH")

# Configure pytesseract
if TESSERACT_PATH:
    import pytesseract
    tesseract_exe = Path(TESSERACT_PATH) / ("tesseract.exe" if IS_WINDOWS else "tesseract")
    pytesseract.pytesseract.tesseract_cmd = str(tesseract_exe)
    add_to_process_path(TESSERACT_PATH, "Tesseract")

# Configure Poppler (needed by pdf2image/unstructured)
if POPPLER_PATH:
    add_to_process_path(POPPLER_PATH, "Poppler")

# Store paths in environment for reference
os.environ["TESSERACT_PATH"] = TESSERACT_PATH or ""
os.environ["POPPLER_PATH"] = POPPLER_PATH or ""

### Importing assets

In [ ]:
def load_data(data_path, filename, url):
    """Downloads a file from a personal Google Drive folder.
    The dataset is stored in the specified data_path directory."""

    # We create the directory in which the dataset will be stored, if not already present
    if not os.path.exists(data_path):
        os.makedirs(data_path)

    save_path = data_path + "/" + filename

    if os.path.exists(save_path):
        print(f"{filename} already exists. Skipping download.")
        return save_path
    else:
        try:
            print("Retrieving the file. This may take a while...")
            gdown.download(url, save_path, quiet=False, fuzzy=True)
            return save_path
        except Exception as e:
            print("An error occurred while downloading the file:", e)
            sys.exit(1)

def extract_data(save_path, data_path, filename):
    """Extracts the content of a compressed dataset (zip or tar)."""

    collection_path = os.path.join(data_path, filename)
    if os.path.exists(collection_path):
        print("Archive already extracted. Skipping extraction.")
        return

    if save_path.endswith(".zip"):
        with zipfile.ZipFile(save_path, "r") as zf:
            zf.extractall(data_path)
        print(f"{filename} extracted.")

    elif save_path.endswith((".tar", ".tar.gz", ".tgz", ".gz")):
        with tarfile.open(save_path) as tf:
            tf.extractall(data_path, filter="data")
        print(f"{filename} extracted.")
    else:
        raise ValueError("Unsupported archive format")

def load_assets(data_path="assets"):
    """ Extracts all the assets required for the project."""
    # Download ProductDescription.pdf from the Drive repository
    load_data(data_path, filename="ProductDescription.pdf", url=URL_PRODUCT_DESCRIPTION)

    # Download and extract DecisionTrees.zip only if the folder doesn't exist
    decision_trees_path = os.path.join(data_path, "DecisionTrees")
    if os.path.exists(decision_trees_path):
        print("DecisionTrees folder already exists. Skipping download and extraction.")
    else:
        extract_data(
            save_path=load_data(
                data_path,
                filename="DecisionTrees.zip",
                url=URL_DECISION_TREES
            ),
            data_path=data_path,
            filename="DecisionTrees"
        )

    # Delete all .zip files in the assets directory to save space
    for file in os.listdir(data_path):
        if file.endswith(".zip"):
            os.remove(os.path.join(data_path, file))

    # Download and extract LLM models from the Drive repository
    # check if in the models folder there are some gguf files
    models_path = "models"
    gguf_files = [f for f in os.listdir(models_path) if f.endswith(".gguf")] if os.path.exists(models_path) else []
    if gguf_files:
        print("LLM models already exist. Skipping download and extraction.")
    else:
        extract_data(
            save_path=load_data(
                ".",
                filename="models.zip",
                url=URL_MODELS
            ),
            data_path=".",
            filename="models"
        )

load_assets()

## Configuration

Parameters and constants needed for our system.

> **Note**: The system uses `llama-cpp-python` for local LLM inference with CUDA acceleration. Make sure to configure the `MODEL_PATH` to point to your GGUF model file.

In [ ]:
# =============================================================================
# REPRODUCIBILITY CONFIGURATION
# =============================================================================

# Set seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =============================================================================
# LLM CONFIGURATION
# =============================================================================

# Directory for models (manually downloaded)
MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

# Model files (must be manually downloaded and placed in MODELS_DIR)
TEXT_MODEL_FILE = "gemma-3-4b-it-q4_0.gguf"
VISION_MODEL_FILE = "gemma-3-4b-it-q4_0.gguf"  # Same model used for vision
VISION_MODEL_MMPROJ = "mmproj-model-f16-4B.gguf"  # CLIP model for vision

# LLM parameters
N_CTX = 8192           # Context window size
N_GPU_LAYERS = -1       # -1 = offload all layers to GPU
TEMPERATURE = 0.1       # Low temperature for deterministic outputs
MAX_NEW_TOKENS = 2048   # Max tokens for responses
VERBOSE = True         # Llama.cpp debug output

# Validate files exist
text_model_path = MODELS_DIR / TEXT_MODEL_FILE
vision_model_path = MODELS_DIR / VISION_MODEL_FILE
mmproj_path = MODELS_DIR / VISION_MODEL_MMPROJ

print("Model files validation:")
print(f"  Text model: {text_model_path} - {'✓' if text_model_path.exists() else '✗ NOT FOUND'}")
print(f"  Vision model: {vision_model_path} - {'✓' if vision_model_path.exists() else '✗ NOT FOUND'}")
print(f"  MMPROJ file: {mmproj_path} - {'✓' if mmproj_path.exists() else '✗ NOT FOUND'}")

# =============================================================================
# FILE PATHS CONFIGURATION
# =============================================================================
PRODUCT_DESCRIPTION_PDF = "./assets/ProductDescription.pdf"
DECISION_TREES_DIR = Path("./assets/DecisionTrees")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

# =============================================================================
# PREPROCESSING & RAG CONFIGURATION
# =============================================================================
PREPROCESSING_CACHE_FILE = CACHE_DIR / "preprocessing_cache.pkl"
PREPROCESSING_HASH_FILE = CACHE_DIR / "preprocessing_hash.txt"

INDEX_PATH = "./data/index"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K_DOCUMENTS = 5
CHUNK_SIZE = 1000

# Print summary
print(f"\nConfiguration:")
print(f"  GPU layers: {N_GPU_LAYERS} ({'all' if N_GPU_LAYERS == -1 else 'partial'})")
print(f"  Context window: {N_CTX}")
print(f"  PDF exists: {os.path.exists(PRODUCT_DESCRIPTION_PDF)}")
print(f"  Decision trees: {len(list(DECISION_TREES_DIR.glob('*.txt')))} files")


## 2. UML Parser

A crucial component of our system is the ability to parse and navigate decision trees encoded in **PlantUML** format. These decision trees, defined by the EN 18031-1 standard, represent the logical flow of compliance requirements. \
One decision tree represents a single requirement. Each tree is made up of nodes and edges, where a node can be either:

- **A question/requirement** that needs to be evaluated against the documentation.
- **A verdict** (PASS, FAIL, NOT APPLICABLE) representing the final evaluation of that requirement.

### Data Structures

In order to navigate a tree, we define three main classes:
- **`Edge`**: Represents a connection between nodes with an optional label
- **`UMLNode`**: Represents a node in the decision tree with its type, text content, and outgoing edges
- **`UMLDecisionTree`**: Wrapper class providing index lookup, serialization, traversal, and printing

In [ ]:
@dataclass
class UMLEdge:
    """Represents an edge in the decision tree graph.
    
    Attributes:
        label: Optional label for the edge (e.g., "Yes", "No")
        target: The UMLNode this edge points to
    """
    label: Optional[str]
    target: "UMLNode"


@dataclass
class UMLNode:
    """A node in the decision tree.

    Attributes:
        id: Unique identifier for the node
        text: The text content of the node (question, statement, or verdict)
        kind: Type of node - one of:
            - "start": synthetic start node
            - "action": plain activity/statement (":" ... ";")
            - "decision": an if-question node
            - "switch": a multi-branch question node
            - "verdict": terminal verdict like PASS/FAIL/NOT APPLICABLE
            - "terminal": explicit end due to 'detach;'
            - "join": implicit merge after if/switch
        edges: List of outgoing edges to other nodes
    """

    id: int
    text: str
    kind: str
    edges: List[UMLEdge] = field(default_factory=list)

    def add_child(self, child: "UMLNode", label: Optional[str] = None) -> None:
        """Add a child node with an optional edge label."""
        self.edges.append(UMLEdge(label=label, target=child))

    def to_dict(self) -> Dict[str, Any]:
        """Convert node to dictionary representation for serialization."""
        return {
            "id": self.id,
            "text": self.text,
            "kind": self.kind,
            "edges": [{"label": e.label, "target": e.target.id} for e in self.edges],
        }


class UMLDecisionTree:
    """Wrapper class for navigating and manipulating parsed decision trees."""
    
    def __init__(self, root: UMLNode) -> None:
        self.root = root
        self.start = root
        self._index: Dict[int, UMLNode] = {}
        self._build_index(root)

    def _build_index(self, node: UMLNode) -> None:
        """Recursively build an index of all nodes by ID."""
        if node.id in self._index:
            return
        self._index[node.id] = node
        for e in node.edges:
            self._build_index(e.target)

    def to_dict(self) -> Dict[str, Any]:
        """Convert tree to dictionary representation for serialization."""
        nodes = {}
        for nid, node in self._index.items():
            nodes[nid] = node.to_dict()
        return {"root": self.root.id, "nodes": nodes}

    def pretty_print(self) -> None:
        """Print a readable outline of the tree structure."""
        seen = set()

        def dfs(n: UMLNode, indent: str = "") -> None:
            if n.id in seen:
                print(f"{indent}[{n.kind}] {n.text} (↩)")
                return
            seen.add(n.id)
            print(f"{indent}[{n.kind}] {n.text} (id={n.id})")
            for e in n.edges:
                lab = f" --{e.label}--> " if e.label else " --> "
                print(f"{indent}{lab}")
                dfs(e.target, indent + "    ")

        dfs(self.root)

    def traverse(self, answer_fn) -> UMLNode:
        """
        Traverse interactively using an answer function that picks the next edge.
        
        The function receives the current node and its edges, and must return the
        index of the chosen edge (0-based), or None to stop.
        
        Returns:
            The last visited node.
        """
        node = self.root
        while True:
            if not node.edges:
                return node
            idx = answer_fn(node, node.edges)
            if idx is None or idx < 0 or idx >= len(node.edges):
                return node
            node = node.edges[idx].target

Now we can define the proper `UMLParser` class that is responsible for parsing PlantUML flowchart syntax and building a navigable tree structure. The parser uses a **stack-based approach** to handle nested control structures ($if/else$, $switch/case$).

The parser maintains a context stack where each context tracks:
- The type of control structure (`"if"` or `"switch"`)
- The control node (decision or switch node)
- A join node for merging branches after the control structure
- The active label for the current branch

This allows proper handling of nested decision structures commonly found in compliance trees.

In [ ]:
class UMLSyntaxError(Exception):
    """Custom exception for UML parsing errors."""
    pass

class UMLParser:
    """
    Minimal PlantUML-flowchart parser tailored for decision trees.
    
    This parser validates structure and builds a navigable tree from PlantUML
    decision tree files (like SUM/TCM compliance examples).

    Supported constructs (single line, trimmed, case-sensitive):
    - @startuml ... @enduml (required)
    - Activity nodes:  ":some text;"  (leading colon and trailing semicolon)
    - if (QUESTION) then (LABEL)
    - else (LABEL)
    - endif
    - switch (QUESTION)
    - case (LABEL)
    - endswitch
    - Verdict lines starting with one of:
        #lightgreen: PASS ...
        #pink: FAIL ...
        #application: NOT APPLICABLE ...
    - detach; (explicit branch termination)
    """

    def __init__(self) -> None:
        self._next_id = 1

    def _new(self, text: str, kind: str) -> UMLNode:
        """Create a new node with auto-incremented ID."""
        n = UMLNode(id=self._next_id, text=text, kind=kind)
        self._next_id += 1
        return n

    def parse_file(self, path: str) -> "UMLDecisionTree":
        """Parse a PlantUML file and return a UMLDecisionTree."""
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        return self.parse_string(content, source=path)

    def parse_string(self, content: str, source: str = "<memory>") -> "UMLDecisionTree":
        """Parse PlantUML content string and return a UMLDecisionTree."""
        # First, normalize multi-line statements by replacing literal \n with space
        content = content.replace("\\n", " ")
        
        lines = [ln.strip() for ln in content.splitlines()]
        if not self._is_plantuml(lines):
            raise UMLSyntaxError(
                f"{source}: missing @startuml/@enduml or malformed PlantUML block"
            )
        # trim to inside of markers and drop empty/comment-only lines
        body = self._extract_body(lines)
        
        # Join lines that are continuations (don't start with keywords and previous line doesn't end properly)
        merged_body = []
        i = 0
        while i < len(body):
            line = body[i]
            # Keywords that should never be merged (they are complete statements)
            complete_keywords = ("if ", "else ", "else(", "endif", "switch ", "case ", "endswitch", ":", "#", "detach", "'", "@")
            # Check if next line is a continuation (doesn't start with keywords)
            while i + 1 < len(body):
                next_line = body[i + 1]
                # If next line doesn't start with a keyword and current line doesn't end with ; or ) and current line is not a complete keyword, merge them
                if (next_line and 
                    not next_line.startswith(complete_keywords) and
                    not line.endswith((";", ")")) and
                    line not in ("endif", "endswitch")):  # Don't merge after endif/endswitch
                    line = line + " " + next_line
                    i += 1
                else:
                    break
            merged_body.append(line)
            i += 1
        
        tokens = [
            ln for ln in merged_body if ln and not ln.startswith("'")
        ]  # ignore PlantUML comments starting with '
        return self._parse_tokens(tokens, source)

    def _is_plantuml(self, lines: List[str]) -> bool:
        """Check if the content is valid PlantUML."""
        try:
            i0 = lines.index("@startuml")
            i1 = len(lines) - 1 - list(reversed(lines)).index("@enduml")
        except ValueError:
            return False
        return i0 < i1

    def _extract_body(self, lines: List[str]) -> List[str]:
        """Extract the body content between @startuml and @enduml."""
        i0 = lines.index("@startuml")
        i1 = len(lines) - 1 - list(reversed(lines)).index("@enduml")
        return [ln for ln in lines[i0 + 1 : i1]]

    # Helpers to recognize forms quickly
    def _starts(self, s: str, prefix: str) -> bool:
        return s.startswith(prefix)

    def _is_activity(self, s: str) -> bool:
        return self._starts(s, ":") and s.endswith(";")

    def _is_if(self, s: str) -> bool:
        return self._starts(s, "if ") and " then " in s and s.endswith(")")

    def _split_if(self, s: str) -> Tuple[str, str]:
        """Parse if (QUESTION) then (LABEL) syntax."""
        try:
            pre, post = s.split(" then ", 1)
            q = pre[len("if ") :].strip()
            label = post.strip()
            if not (q.startswith("(") and q.endswith(")")):
                raise ValueError
            if not (label.startswith("(") and label.endswith(")")):
                raise ValueError
            return q[1:-1].strip(), label[1:-1].strip()
        except Exception:
            raise UMLSyntaxError(f"Malformed if/then line: {s}")

    def _is_else(self, s: str) -> bool:
        return self._starts(s, "else ") and s.endswith(")")

    def _split_else(self, s: str) -> str:
        """Parse else (LABEL) syntax."""
        lab = s[len("else ") :].strip()
        if not (lab.startswith("(") and lab.endswith(")")):
            raise UMLSyntaxError(f"Malformed else line: {s}")
        return lab[1:-1].strip()

    def _is_switch(self, s: str) -> bool:
        return self._starts(s, "switch ") and s.endswith(")")

    def _split_switch(self, s: str) -> str:
        """Parse switch (QUESTION) syntax."""
        q = s[len("switch ") :].strip()
        if not (q.startswith("(") and q.endswith(")")):
            raise UMLSyntaxError(f"Malformed switch line: {s}")
        return q[1:-1].strip()

    def _is_case(self, s: str) -> bool:
        return self._starts(s, "case ") and s.endswith(")")

    def _split_case(self, s: str) -> str:
        """Parse case (LABEL) syntax."""
        lab = s[len("case ") :].strip()
        if not (lab.startswith("(") and lab.endswith(")")):
            raise UMLSyntaxError(f"Malformed case line: {s}")
        return lab[1:-1].strip()

    def _is_verdict(self, s: str) -> Optional[Tuple[str, str]]:
        """Check if line is a verdict and return (kind, text) if so."""
        # Check for verdict lines with or without space before colon
        # e.g., "#lightgreen:" or "#application :"
        for prefix_base, kind in (
            ("#lightgreen", "PASS"),
            ("#pink", "FAIL"),
            ("#application", "NOT APPLICABLE"),
        ):
            # Try with and without space before colon
            for prefix in [prefix_base + ":", prefix_base + " :"]:
                if s.startswith(prefix):
                    # Extract text after the colon
                    colon_idx = s.index(":")
                    return kind, s[colon_idx + 1:].strip()
        return None

    def _parse_tokens(self, tokens: List[str], source: str) -> "UMLDecisionTree":
        """Main parsing logic - builds the decision tree from tokens."""
        start = self._new(text="START", kind="start")
        current: UMLNode = start
        ctx_stack: List[Tuple[str, UMLNode, UMLNode, Optional[str]]] = []
        # tuple: (kind, control_node, join_node, active_label)
        #   - active_label: branch label for next child emitted under current control

        def ensure_join(control: UMLNode) -> UMLNode:
            # find existing join belonging to top-of-stack or create a new one
            join = self._new(text=f"JOIN after {control.kind}", kind="join")
            return join

        i = 0
        while i < len(tokens):
            line = tokens[i]
            i += 1

            if line == "":
                continue

            if line == "endif":
                if not ctx_stack or ctx_stack[-1][0] != "if":
                    raise UMLSyntaxError(f"{source}: 'endif' without matching 'if'")
                _, control, _, _ = ctx_stack.pop()
                join = ensure_join(control)
                # Link if control to join if there are branches without explicit termination
                control.add_child(join, label=None)  # unlabeled fall-through
                current = join
                continue

            if line == "endswitch":
                if not ctx_stack or ctx_stack[-1][0] != "switch":
                    raise UMLSyntaxError(
                        f"{source}: 'endswitch' without matching 'switch'"
                    )
                _, control, _, _ = ctx_stack.pop()
                join = ensure_join(control)
                control.add_child(join, label=None)
                current = join
                continue

            # Handle both "detach;" and "detach" (with or without semicolon)
            if line == "detach;" or line == "detach":
                term = self._new(text="DETACH", kind="terminal")
                current.add_child(term, label=None)
                # After detach, set current to the terminal node
                # The endif/endswitch will handle proper cleanup
                current = term
                continue

            if self._is_activity(line):
                text = line[1:-1].strip()  # remove leading ':' and trailing ';'
                node = self._new(text=text, kind="action")
                current.add_child(node)
                current = node
                continue

            if self._is_if(line):
                q, then_label = self._split_if(line)
                decision = self._new(text=q, kind="decision")
                current.add_child(decision)
                join = self._new(text="JOIN after IF", kind="join")
                ctx_stack.append(("if", decision, join, then_label))
                # move current to decision; next content belongs to 'then' branch
                current = decision
                # mark active label so that first emitted node becomes that branch
                ctx_stack[-1] = ("if", decision, join, then_label)
                continue

            if self._is_else(line):
                if not ctx_stack or ctx_stack[-1][0] != "if":
                    raise UMLSyntaxError(f"{source}: 'else' without matching 'if'")
                kind, decision, join, _ = ctx_stack[-1]
                # switch current back to decision to attach an alternative branch
                current = decision
                new_label = self._split_else(line)
                ctx_stack[-1] = (kind, decision, join, new_label)
                continue

            if self._is_switch(line):
                q = self._split_switch(line)
                sw = self._new(text=q, kind="switch")
                current.add_child(sw)
                join = self._new(text="JOIN after SWITCH", kind="join")
                ctx_stack.append(("switch", sw, join, None))
                current = sw
                continue

            if self._is_case(line):
                if not ctx_stack or ctx_stack[-1][0] != "switch":
                    raise UMLSyntaxError(f"{source}: 'case' without matching 'switch'")
                kind, sw, join, _ = ctx_stack[-1]
                current = sw
                lab = self._split_case(line)
                ctx_stack[-1] = (kind, sw, join, lab)
                continue

            verdict = self._is_verdict(line)
            if verdict:
                vkind, rest = verdict
                node = self._new(text=f"{vkind}: {rest}", kind="verdict")
                # If inside a control with an active branch label, attach under that label
                if ctx_stack and ctx_stack[-1][3] is not None:
                    label = ctx_stack[-1][3]
                    ctx_stack[-1] = (
                        ctx_stack[-1][0],
                        ctx_stack[-1][1],
                        ctx_stack[-1][2],
                        None,
                    )
                    ctx_stack[-1][1].add_child(node, label=label)
                else:
                    current.add_child(node)
                current = node
                continue

            # Default: a raw statement in a branch head -> treat like activity
            if ctx_stack and ctx_stack[-1][3] is not None:
                # First node of the branch: attach with label
                label = ctx_stack[-1][3]
                ctx_stack[-1] = (
                    ctx_stack[-1][0],
                    ctx_stack[-1][1],
                    ctx_stack[-1][2],
                    None,
                )
                node = self._new(text=line, kind="action")
                ctx_stack[-1][1].add_child(node, label=label)
                current = node
                continue

            # If we reach here, the token is unsupported
            raise UMLSyntaxError(f"{source}: unsupported or malformed line: {line}")

        # Unwound stack means unmatched endif/endswitch
        if ctx_stack:
            kinds = ", ".join(k for k, *_ in ctx_stack)
            raise UMLSyntaxError(f"{source}: unmatched blocks left open: {kinds}")

        return UMLDecisionTree(start)

We can now try to load and inspect a decision tree from the actual files:

In [ ]:
# Example: Parse and inspect a decision tree from a PlantUML file
parser = UMLParser()

# Load one of the decision tree files
example_file = f"{DECISION_TREES_DIR}/NMM-1.txt"
tree = parser.parse_file(example_file)

# Print a human-readable outline of the tree structure
print(f"Decision tree loaded from: {example_file}\n")
print("=" * 60)
tree.pretty_print()
print("=" * 60)

Se andiamo a visualizzare il suddetto albero tramite un tool di visualizzazione PlantUML otteniamo la seguente immagine:

![Decision Tree Example](./assets/images/NMM-1.png)

---

## LLM Client Classes

We define classes for interacting with LLMs using `llama-cpp-python`:
- **`LLMClient`**: Abstract base class defining the interface
- **`LlamaCppClient`**: Direct Python bindings to llama.cpp with CUDA support

In [ ]:
class LLMError(Exception):
    """Base exception for LLM-related errors."""
    pass


class LLMClient(ABC):
    """Abstract base class for LLM clients."""

    # Class-level counter
    call_count = 0
    
    @abstractmethod
    def generate(self, prompt: str, system_prompt: str = "", max_tokens: int = MAX_NEW_TOKENS) -> str:
        """Generate text response from a prompt."""
        pass
    
    @abstractmethod
    def generate_with_image(self, prompt: str, image_bytes: bytes, system_prompt: str = "", max_tokens: int = MAX_NEW_TOKENS) -> str:
        """Generate text response from a prompt with an image (multimodal)."""
        pass

    @classmethod
    def get_call_count(cls) -> int:
        """Return the total number of LLM calls made."""
        return cls.call_count
    
    @classmethod
    def reset_call_count(cls):
        """Reset the call counter to zero."""
        cls.call_count = 0


class LlamaCppClient(LLMClient):
    """Client for llama-cpp-python."""

    def __init__(self, model_file: str, clip_model_file: Optional[str] = None,
                n_ctx: int = 4096, n_gpu_layers: int = -1,
                temperature: float = 0.1, seed: int = 42, verbose: bool = True):
        """
        Initialize the llama.cpp client with a single model instance.

        Args:
            model_file: Path to GGUF model file (in MODELS_DIR)
            clip_model_file: Path to CLIP/MMPROJ file for vision (optional)
            n_ctx: Context window size
            n_gpu_layers: Number of layers to offload to GPU (-1 = all)
            temperature: Sampling temperature
            seed: Random seed
            verbose: Enable verbose output
        """
        # Force garbage collection before loading new model to prevent memory issues
        gc.collect()
        
        self.temperature = temperature
        self.model_path = MODELS_DIR / model_file
        self.is_vision = clip_model_file is not None
        
        # Validate model file exists
        if not self.model_path.exists():
            raise LLMError(f"Model file not found: {self.model_path}")
        
        # Load model
        if self.is_vision:
            # Vision model
            clip_path = MODELS_DIR / clip_model_file
            if not clip_path.exists():
                raise LLMError(f"CLIP model file not found: {clip_path}")
            
            print(f"🖼️  Loading vision model: {model_file}")
            print(f"   CLIP model: {clip_model_file}")
            
            try:
                # Use Llava15ChatHandler
                chat_handler = Llava15ChatHandler(clip_model_path=str(clip_path))
                self.llm = Llama(
                    model_path=str(self.model_path),
                    chat_handler=chat_handler,
                    n_ctx=n_ctx,
                    n_gpu_layers=n_gpu_layers,
                    seed=seed,
                    verbose=verbose
                )
                print(f"   Vision model loaded (ctx={n_ctx}, gpu_layers={n_gpu_layers})")
            except Exception as e:
                raise LLMError(f"Failed to load vision model: {e}")
        else:
            # Text-only model
            print(f"📝 Loading text model: {model_file}")
            
            try:
                self.llm = Llama(
                    model_path=str(self.model_path),
                    n_ctx=n_ctx,
                    n_gpu_layers=n_gpu_layers,
                    seed=seed,
                    verbose=verbose
                )
                print(f"   ✓ Text model loaded (ctx={n_ctx}, gpu_layers={n_gpu_layers})")
            except Exception as e:
                raise LLMError(f"Failed to load text model: {e}")
    
    def generate(self, prompt: str, system_prompt: str = "", max_tokens: int = MAX_NEW_TOKENS) -> str:
        """Generate text response using chat completion format."""
        try:
            if system_prompt:
                full_prompt = f"<start_of_turn>user \n{system_prompt}\n\n{prompt}<end_of_turn>\n<start_of_turn>model"
            else:
                full_prompt = f"<start_of_turn>user \n{prompt}<end_of_turn>\n<start_of_turn>model"

            messages = []
            #if system_prompt:
            #    messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": full_prompt})
            
            result = self.llm.create_chat_completion(
                messages=messages,
                max_tokens=max_tokens,
                temperature=self.temperature
            )
            LLMClient.call_count += 1
            return result["choices"][0]["message"]["content"].strip()
        except Exception as e:
            raise LLMError(f"Text generation failed: {e}")
    
    def generate_with_image(self, prompt: str, image_bytes: bytes, system_prompt: str = "", max_tokens: int = MAX_NEW_TOKENS) -> str:
        """Generate text response with an image (multimodal)."""
        if not self.is_vision:
            raise LLMError("This model does not support vision. Initialize with clip_model_file for vision support.")
        
        try:
            if system_prompt:
                full_prompt = f"<start_of_turn>user \n{system_prompt}\n\n{prompt}<end_of_turn>\n<start_of_turn>model"
            else:
                full_prompt = f"<start_of_turn>user \n{prompt}<end_of_turn>\n<start_of_turn>model"
            
            image_b64 = base64.b64encode(image_bytes).decode('utf-8')

            result = self.llm.create_chat_completion(
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                        {"type": "text", "text": full_prompt}
                    ]
                }],
                max_tokens=max_tokens,
                temperature=self.temperature
            )
            LLMClient.call_count += 1
            return result["choices"][0]["message"]["content"].strip()
        except Exception as e:
            raise LLMError(f"Vision generation failed: {e}")
    
    #def __del__(self):
    #    """Cleanup when object is destroyed."""
    #    if hasattr(self, 'llm'):
    #        del self.llm
    #        print("   Model unloaded from memory")


### Testing the model

In [ ]:
if 'llm_client' in globals():
    del llm_client  # Cleanup previous instance # type: ignore

llm_client = LlamaCppClient(
    model_file=TEXT_MODEL_FILE,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    temperature=TEMPERATURE,
    seed=RANDOM_SEED,
    verbose=True
)

res = llm_client.generate("Hi from the Multimedia Information Retrieval and Computer Vision project at University of Pisa! Who are you? Answer briefly.")
print("LLM test response:", res)

---

## 4. Document Preprocessing

One of the critical challenges in processing product documentation is handling **complex visual elements** such as images, diagrams, and tables. Standard text extraction often fails to capture the semantic content of these elements, and yet this kind of content can bring crucial context for compliance verification.

We decided to use a **hybrid preprocessing pipeline** that combines:
1. Smart **partitioning** and element extraction using [**Unstructured**](https://github.com/Unstructured-IO/unstructured), a library for handling the partitioning, chunking and, in general, preprocessing of several file formats specifically for RAG systems.
2. A **Vision LLM** for understanding and describing visual content such as images, schemes and tables.

We do this because:

- **Images**: May contain important diagrams, flowcharts, or technical illustrations
- **Tables**: Structured data that loses meaning when extracted as plain text

For simplicity we focus only on PDF documents.

### Details on Partitioning

Before the full preprocessing pipeline, let's analyze how Unstructured partitions pages.

In [ ]:
SAMPLE_PAGE_NUM = 14

# Extract sample page for analysis
doc = fitz.open(PRODUCT_DESCRIPTION_PDF)    # Open the PDF document
total_pages_pdf = len(doc)  # Get total number of pages

if SAMPLE_PAGE_NUM < 1 or SAMPLE_PAGE_NUM > total_pages_pdf:
    # check page number validity
    print(f"Warning: Page {SAMPLE_PAGE_NUM} out of range (1-{total_pages_pdf}). Defaulting to 1.")

# Exctract the specified page and save to a temporary PDF
temp_doc = fitz.open() # Create a new PDF for the sample page
temp_doc.insert_pdf(doc, from_page=SAMPLE_PAGE_NUM-1, to_page=SAMPLE_PAGE_NUM-1)    # Insert the sample page
temp_page_file = f"test_page_{SAMPLE_PAGE_NUM}.pdf"   # Temporary file path
temp_doc.save(temp_page_file)
doc.close()

# Partition with same settings as main pipeline
print(f"Partitioning page {SAMPLE_PAGE_NUM} with Unstructured...")
page_elements = partition_pdf(
    filename=temp_page_file,
    strategy="hi_res",  # Use TESERACT OCR for high-res layout analysis
    include_page_breaks=True,
    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    starting_page_number=SAMPLE_PAGE_NUM,
    extract_image_block_output_dir="cache/figures",
    languages=["eng"],
)

# Analyze element types
print(f"\nTotal elements extracted: {len(page_elements)}")  # total elements
element_types = Counter(type(elem).__name__ for elem in page_elements) # count by type
print("\nElement types:")
for elem_type, count in sorted(element_types.items()):
    print(f"  {elem_type}: {count}")

# Show sample metadata
print("\nSample element metadata:")
if page_elements:
    sample = page_elements[5]
    print(f"  Type: {type(sample).__name__}")
    print(f"  Text: {sample.text[:100] if hasattr(sample, 'text') and sample.text else 'N/A'}")
    if hasattr(sample, 'metadata'):
        metadata_dict = sample.metadata.to_dict()
        print(f"  Metadata keys: {list(metadata_dict.keys())}")
        if 'coordinates' in metadata_dict:
            print(f"  Coordinates: {metadata_dict['coordinates']}")

We found several elements in the test page and we note that it seems there are more than 180 images in that page, which is quite strange.

In order to verify the quality of the partitioning - in terms of sections found, types of elements and position of each element - we proceed visualizing the test page showing the bounding boxes of each element found, colored by type.

In [ ]:
def unstructured_bbox_to_pixel_rect(coords, page_width, page_height, pix_width, pix_height):
    # Convert Unstructured coordinates to pixel rectangle on the image
    points = coords.points
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    # Prefer Unstructured coordinate system dimensions if available
    base_w = getattr(getattr(coords, "system", None), "width", None) or page_width
    base_h = getattr(getattr(coords, "system", None), "height", None) or page_height

    # Compute scaling factors
    scale_x = pix_width / base_w
    scale_y = pix_height / base_h

    # coords are top-left origin
    x = x_min * scale_x
    y = y_min * scale_y
    w = (x_max - x_min) * scale_x
    h = (y_max - y_min) * scale_y

    return int(round(x)), int(round(y)), int(round(w)), int(round(h))

# Render page as image
page = temp_doc[0]  # single page document
pix = page.get_pixmap(dpi=300)  # Higher DPI for better visualization
img_bytes = pix.tobytes("png")
page_image = PILImage.open(io.BytesIO(img_bytes))
page_width = page.rect.width
page_height = page.rect.height

# Create figure with matplotlib
fig, ax = plt.subplots(1, 1, figsize=(6, 16))  # adjust size as needed
ax.imshow(page_image)
ax.axis('off')

# Color mapping for element types
color_map = {
    'Image': 'blue',
    'Table': 'red',
    'Text': 'gray',
    'Title': 'green',
    'NarrativeText': 'yellow',
    'ListItem': 'cyan',
    'Header': 'magenta',
    'FigureCaption': 'orange',
    'PageBreak': 'purple',
}

# Overlay bounding boxes for each element
for elem in page_elements:
    coords = getattr(getattr(elem, "metadata", None), "coordinates", None)  # get coordinates
    if not coords or not hasattr(coords, "points") or not coords.points:
        continue

    # Convert to pixel rectangle
    x, y, w, h = unstructured_bbox_to_pixel_rect(
        coords,
        page_width, page_height,
        pix.width, pix.height
    )

    # Clamp to image bounds
    if w <= 0 or h <= 0:
        continue
    x = max(0, min(x, pix.width - 1))
    y = max(0, min(y, pix.height - 1))
    w = max(1, min(w, pix.width - x))
    h = max(1, min(h, pix.height - y))

    elem_type = type(elem).__name__
    color = color_map.get(elem_type, "lightgreen")

    rect = patches.Rectangle((x, y), w, h, linewidth=1.2, edgecolor=color, facecolor='none', alpha=0.7)
    ax.add_patch(rect)


plt.title("Partitioning Visualization", fontsize=14)
# add legend
plt.legend(
    handles=[patches.Patch(color=color_map[elem_type], label=elem_type) for elem_type in color_map], 
    fontsize=6, 
    loc='lower left'
)
plt.tight_layout()
plt.show()

As we can notice, Unstructured is pretty good in extracting and categorizing different sections of the PDF page, yet it's not perfect. \
We can see a few misclassifications like the footer and the left table that are both classified as simple text.

Regardless, the overall quality is acceptable for the purpose of this project.

> Note the several small blue bounding boxes over the PCB image: those are artifacts that explain why Unstructured found so many images in that page. Also, in the caching folder where Unstructured saves the images extracted from the PDF, you can see that there are indeed many microscopic images that are absolutely useless. We'll need to keep that in mind and filter them out during the processing.

In particular, the labeling of the different sections allows us to filter only the ones we are actually interested in.

### Visual Elements description with VLM

Now that we verified the partitioning quality, we need to define the functions we'll be using to ask the Vision LLM to describe visual elements.

In [ ]:
def bytes_to_pil_image(image_bytes: bytes) -> PILImage.Image:
    # Convert raw image bytes to PIL Image.
    return PILImage.open(io.BytesIO(image_bytes)).convert("RGB")


def pil_to_png_bytes(image: PILImage.Image) -> bytes:
    # Convert PIL Image to PNG bytes.
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


def extract_image_with_vision(image_bytes: bytes) -> str:
    # Use Vision LLM to extract text and describe diagrams/charts.
    # TODO: refine prompt engineering.
    system_prompt = """
        You are a vision-language model used in a technical documentation analysis pipeline.

        Your task is to extract structured information from technical images to support a Retrieval-Augmented Generation (RAG) system.

        You MUST strictly follow these principles:
        - Describe only what is explicitly visible in the image.
        - Do NOT infer, assume, or guess missing information.
        - Do NOT use external knowledge or domain expertise.
        - If information is not clearly visible, omit it.
        - Prefer omission over speculation.

        Output length and style constraints:
        - Be concise and information-dense.
        - Do NOT generate verbose explanations.
        - Do NOT repeat the same information using different wording.
        - Do NOT add filler text, rephrasings, or redundant statements.
        - Include only information that adds new factual value.

        Your output must be:
        - Structured
        - Compact
        - Neutral and factual
        - Strictly limited to the information requested by the user prompt

        The image may contain:
        - Product schematics
        - Block diagrams
        - Wiring diagrams
        - Mechanical or technical drawings

        You are NOT allowed to:
        - Explain how the system works beyond what is visually shown
        - Interpret symbols unless their meaning is explicitly labeled
        - Expand abbreviations unless written in full in the image
        - Add conclusions, summaries, or contextual commentary

        Your goal is to produce a minimal, high-precision visual description
        optimized for indexing and retrieval in a RAG pipeline.
    """

    prompt = """
        Analyze the image using the following structure: 
        1. Image type (e.g., block diagram, wiring schematic, mechanical drawing)
        2. Product or subsystem name (only if explicitly visible)
        3. List of main components:
            - Component name or label
            - Apparent role or function (only if visually implied)
        4. Connections or interactions between components (if shown)
        Rules:
        - Do not guess.
        - Do not add external knowledge.
        - Base everything strictly on what is visible in the image.
        - Be not so long: your output has to be used for a RAG system
    
    """
    
    # Convert to PNG to ensure consistent format
    try:
        image = bytes_to_pil_image(image_bytes)
        png_bytes = pil_to_png_bytes(image)
    except Exception as e:
        raise LLMError(f"Failed to process image: {e}")
    
    return llm_client.generate_with_image(prompt, png_bytes, system_prompt=system_prompt)


def extract_table_with_vision(image_bytes: bytes) -> str:
    # Use Vision LLM to extract and structure table data.
    # TODO: refine prompt engineering.
    system_prompt = """
        You are a vision-language model used in a technical documentation analysis pipeline.

        Your task is to extract structured, factual information from images containing
        technical tables (e.g. pinout tables, signal mappings, connector matrices)
        to support a Retrieval-Augmented Generation (RAG) system.

        You MUST strictly follow these principles:
        - Extract only what is explicitly visible in the table.
        - Do NOT infer electrical behavior, protocol meaning, or signal function.
        - Do NOT assume relationships beyond what is explicitly written.
        - Do NOT use external knowledge.
        - If a cell is unclear, unreadable, or empty, omit it.
        - Prefer omission over speculation.

        Table-specific rules:
        - Preserve the table structure as shown (columns, headers, rows).
        - Keep labels exactly as written (case-sensitive, no expansion of abbreviations).
        - Do NOT normalize or reinterpret signal names.
        - Do NOT merge or deduce equivalences between signals.
        - Treat each row as an independent factual entry.

        Output length and style constraints:
        - Be concise and information-dense.
        - Do NOT generate verbose explanations.
        - Do NOT repeat information already stated.
        - Do NOT add filler text or restatements.
        - Include only data extracted from the table.

        Your output must be:
        - Structured
        - Compact
        - Neutral and factual
        - Strictly limited to the information requested by the user prompt

        You are NOT allowed to:
        - Explain the purpose of the table
        - Interpret pin functions or electrical roles
        - Add conclusions or summaries
        - Add context not explicitly present in the image

        Your goal is to produce a minimal, high-precision transcription
        of the table content, optimized for indexing and retrieval
        in a RAG pipeline.
    """
    
    prompt = "This is a table. Extract all data and present it as structured text with clear headers and rows. Preserve all information and relationships."
    
    # Convert to PNG to ensure consistent format
    try:
        image = bytes_to_pil_image(image_bytes)
        png_bytes = pil_to_png_bytes(image)
    except Exception as e:
        raise LLMError(f"Failed to process table image: {e}")
    
    return llm_client.generate_with_image(prompt, png_bytes, system_prompt=system_prompt)

Let's try to describe an image and a table from our test page using the Vision LLM.

In [ ]:
# Free text model and load vision model
if 'llm_client' in globals():
    del llm_client

llm_client = LlamaCppClient(
    model_file=VISION_MODEL_FILE,
    clip_model_file=VISION_MODEL_MMPROJ,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    temperature=TEMPERATURE,
    seed=RANDOM_SEED,
    verbose=VERBOSE
)

In [ ]:

# Get images and tables from temp page
images = [elem for elem in page_elements if isinstance(elem, Image)]
tables = [elem for elem in page_elements if isinstance(elem, Table)]

# Test with dog image
image_description = None
img = PILImage.open(images[1].metadata.image_path)
print("\nGenerating image description...")
image_description = extract_image_with_vision(pil_to_png_bytes(img))

# Pick first table
tbl = None
table_description = None
if tables and hasattr(tables[0].metadata, 'image_path'):
    tbl = PILImage.open(tables[0].metadata.image_path)
    print("Generating table description...")
    table_description = extract_table_with_vision(pil_to_png_bytes(tbl))


In [ ]:
# Example: Image + VLM Description
if img:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [1, 3]}, )
    
    ax1.imshow(img)
    ax1.axis('off')
    ax1.set_title("Extracted Image", fontweight='bold')

    ax2.text(0.05, 0.95, "VLM Description:", fontweight='bold', va='top', transform=ax2.transAxes)
    ax2.text(0.05, 0.88, image_description.strip('\n'), fontsize=9, va='top', wrap=True, transform=ax2.transAxes)
    ax2.axis('off')
    
    plt.show()
else:
    print("No image available")

In [ ]:
# Example: Table + VLM Description
if tbl:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1, 2]})
    
    ax1.imshow(tbl)
    ax1.axis('off')
    ax1.set_title("Extracted Table", fontweight='bold')
    
    ax2.text(0.05, 0.95, "VLM Description:", fontweight='bold', va='top', transform=ax2.transAxes)
    ax2.text(0.05, 0.88, table_description, fontsize=9, va='top', wrap=True, transform=ax2.transAxes)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No table available")

In [ ]:
# Clean up
temp_doc.close()    # Close temporary document
os.remove(temp_page_file)   # Clean up temporary document
shutil.rmtree("cache/figures")  # Clean up extracted images

### Main PDF Preprocessing Function

The `preprocess_pdf` function orchestrates the entire preprocessing pipeline:

1. Opens the PDF
2. Partitions each page
3. Enriches **Images** and **Tables** using the Vision LLM
4. Tracks processed images by xref to avoid duplicates
5. Returns a list of processed elements with type, content, and page metadata

In [ ]:
def preprocess_pdf(pdf_path: str) -> Tuple[List[Any], List[Dict[str, Any]]]:
    # Process the entire document at once to ensure correct image naming convention.

    print(f"Processing PDF: {pdf_path}")

    # Configuration for filtering noise
    MIN_CROP_AREA_PX = 20_000    # filter out small images (in pixel-area)

    # Partition with Unstructured
    print("Partitioning full document... (this make take a while, please wait)")
    
    # Note: Unstructured does not provide a native progress bar for single-file partitioning
    elements = partition_pdf(
        filename=pdf_path,
        strategy="hi_res",  # Use hi_res for tables and images
        include_page_breaks=True,
        infer_table_structure=True,  # Extract table structure
        extract_image_block_types=["Image", "Table"],  # Extract embedded images and tables
        starting_page_number=1,
        extract_image_block_output_dir="cache/figures",
        languages=["eng"],
    )

    print(f"Partitioning complete. Found {len(elements)} raw elements.")

    # Process elements
    processed_elements = []
    doc = fitz.open(pdf_path)  # Keep open for fallback image extraction

    # Track processed images to avoid duplicates
    skipped_small_images = 0
    element_id = 0

    for element in tqdm(elements, desc="Processing elements"):
        element_type = type(element)

        # Handle images with Vision LLM
        if element_type == Image:
            try:
                # Check if image path exists
                if not hasattr(element.metadata, 'image_path'):
                    continue
                
                img = PILImage.open(element.metadata.image_path)
                
                # Filter out small images (noise)
                img_area = img.width * img.height
                if img_area < MIN_CROP_AREA_PX:
                    skipped_small_images += 1
                    continue
                
                # Extract image description with Vision LLM
                image_bytes = pil_to_png_bytes(img)
                image_description = extract_image_with_vision(image_bytes)
                
                page_num = element.metadata.page_number
                processed_elements.append({
                    'id': element_id,
                    'type': 'image',
                    'content': image_description,
                    'page': page_num
                })
                element_id += 1
            except Exception as e:
                print(f"Error extracting image (non-fatal): {e}")
                continue

        # Handle tables with Vision LLM
        elif element_type == Table:
            try:
                # Check if image path of the table exists
                if not hasattr(element.metadata, 'image_path'):
                    raise ValueError("Table image path missing in metadata.")
                
                tbl = PILImage.open(element.metadata.image_path)
                
                # Filter out small table images
                tbl_area = tbl.width * tbl.height
                if tbl_area < MIN_CROP_AREA_PX:
                    raise ValueError("Table image too small.")
                
                # Extract table description with Vision LLM
                table_bytes = pil_to_png_bytes(tbl)
                table_description = extract_table_with_vision(table_bytes)
                
                page_num = element.metadata.page_number
                processed_elements.append({
                    'id': element_id,
                    'type': 'table',
                    'content': table_description,
                    'page': page_num
                })
                element_id += 1
            except Exception as e:
                print(f"Error processing table (non-fatal): {e}")
                page_num = element.metadata.page_number
                if hasattr(element, 'text') and element.text:
                    processed_elements.append({
                        'id': element_id,
                        'type': 'table',
                        'content': element.text,
                        'page': page_num
                    })
                    element_id += 1

        # Handle regular text elements
        elif element_type in [Text, Title, NarrativeText, ListItem, FigureCaption]:
            if hasattr(element, 'text') and element.text and element.text.strip():
                processed_elements.append({
                    'id': element_id,
                    'type': 'text',
                    'content': element.text,
                    'page': element.metadata.page_number
                })
                element_id += 1
    doc.close()
    print(f"Processed {len(processed_elements)} relevant elements")
    if skipped_small_images > 0:
        print(f"Skipped {skipped_small_images} tiny crops (<{MIN_CROP_AREA_PX} px²)")
    return elements, processed_elements

### Execute Preprocessing

Since the preprocessing phase is computationally expensive due to the Vision LLM calls, to avoid re-processing the same PDF multiple times, we implement a caching mechanism based on file hashing.

Let's define a function to compute the SHA256 hash of a file.

In [ ]:
def compute_file_hash(file_path: str) -> str:
    # Compute SHA256 hash of a file.
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()

And now we can actually run the preprocessing:

In [ ]:
# Check if valid preprocessing cache exists
current_hash = compute_file_hash(PRODUCT_DESCRIPTION_PDF)
cache_valid = (
    PREPROCESSING_CACHE_FILE.exists() 
    and PREPROCESSING_HASH_FILE.exists() 
    and PREPROCESSING_HASH_FILE.read_text().strip() == current_hash
)

if cache_valid: 
    # Load preprocessed elements from cache
    print(f"Cache valid (hash: {current_hash[:16]}...)")
    with open(PREPROCESSING_CACHE_FILE, 'rb') as f:
        processed_elements = pickle.load(f)
    print(f"Loaded {len(processed_elements)} elements from cache")
else:
    print("Cache miss - running full preprocessing with Vision LLM...")
    # Here now we store also the original raw elements
    raw_elements, processed_elements = preprocess_pdf(PRODUCT_DESCRIPTION_PDF)
    
    # Save to cache
    with open(PREPROCESSING_CACHE_FILE, 'wb') as f:
        pickle.dump(processed_elements, f)
    PREPROCESSING_HASH_FILE.write_text(current_hash)
    print(f"✓ Cached {len(processed_elements)} elements (hash: {current_hash[:16]}...)")

# Show summary
type_counts = Counter(elem['type'] for elem in processed_elements)
print(f"\nTotal: {len(processed_elements)} elements")
print("By type:", dict(type_counts))

---

## 5. RAG System

After preprocessing, we need to:
1. **Split** the extracted content into manageable chunks
2. **Embed** each chunk using a sentence transformer model
3. **Index** the embeddings for efficient similarity search

### Why Chunking?

Large documents cannot be processed as a whole due to:
- LLM context window limitations
- Need for precise, focused retrieval
- Memory constraints when computing embeddings

Our chunking strategy maintains page-level metadata to enable source tracking.

### The RAG System

The `RAGSystem` class encapsulates the retrieval pipeline:
- Uses **sentence-transformers** for embedding generation
- Stores embeddings in the **ChromaDB** for efficient vector similarity search
- Supports configurable top-k retrieval

### Why ChromaDB?

We chose **ChromaDB** as our vector database primarily to **avoid reprocessing costs**. Our pipeline uses a Vision LLM to generate textual descriptions for images and tables, which is a computationally expensive operation that can take several minutes per document. ChromaDB's persistent storage allows us to run this expensive preprocessing once and reuse the indexed embeddings across multiple sessions, rather than repeating it on every execution.

ChromaDB is lightweight and runs entirely as a local library. It supports arbitrary metadata alongside embeddings (page numbers, element types), enabling filtered retrieval and source tracking, which is essential for compliance checking. Under the hood, it uses the HNSW algorithm for efficient approximate nearest neighbor search.

### Alternative Chuncking strategy (using Unstructured)

According to the unstructured documentation (https://docs.unstructured.io/open-source/core-functionality/chunking), the <i> by title <i> chuncking strategy preserves section boundaries, in the sense that a single text will never contain text that occoured in different sections.

In [ ]:
class RAGSystem:
    """
    RAG system using sentence transformers and ChromaDB.
    
    This class handles:
    - Document indexing with hash-based caching to avoid reprocessing
    - Chunking of preprocessed content
    - Embedding generation using sentence-transformers
    - Vector storage and indexing with ChromaDB
    - Similarity-based document retrieval
    
    Attributes:
        embedding_model: The sentence transformer model for embeddings
        client: ChromaDB client for vector storage
        collection: The ChromaDB collection storing document embeddings
    """

    def __init__(self, embedding_model_name: str = EMBEDDING_MODEL):
        """Initialize the RAG system with specified embedding model."""
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.client = chromadb.PersistentClient(path=INDEX_PATH, settings=Settings(anonymized_telemetry=False))
        self.collection = None

    # -------------------------------------------------------------------------
    # Hash-based caching methods
    # -------------------------------------------------------------------------

    def _get_collection_hash(self, collection_name: str) -> Optional[str]:
        """
        Get the stored document hash from collection metadata.
        
        Args:
            collection_name: Name of the ChromaDB collection
            
        Returns:
            Stored hash string or None if collection doesn't exist
        """
        existing_collections = self.client.list_collections()
        for col in existing_collections:
            if col.name == collection_name:
                collection = self.client.get_collection(name=collection_name)
                metadata = collection.metadata or {}
                return metadata.get("source_file_hash")
        return None

    def _is_document_indexed(self, file_path: str, collection_name: str) -> bool:
        """
        Check if a document is already indexed with matching hash.
        
        Args:
            file_path: Path to the source document
            collection_name: Name of the ChromaDB collection
            
        Returns:
            True if document is already indexed with same hash
        """
        current_hash = compute_file_hash(file_path)  # Uses standalone function
        stored_hash = self._get_collection_hash(collection_name)
        return stored_hash is not None and current_hash == stored_hash

    # -------------------------------------------------------------------------
    # Chunking
    # -------------------------------------------------------------------------
    
    @staticmethod
    def _chunk_elements(elements: List[Dict[str, Any]], chunk_size: int) -> List[Dict[str, Any]]:
        """
        Split processed elements into chunks for RAG retrieval.
        
        This method aggregates consecutive elements until the chunk size limit
        is reached, preserving page metadata for source tracking.
        
        Args:
            elements: List of processed document elements
            chunk_size: Maximum characters per chunk
            
        Returns:
            List of chunks with id, content, and page metadata
        """
        chunks = []
        current_chunk = ""
        current_page = None
        chunk_id = 0

        for element in elements:
            content = element['content']
            page = element.get('page')

            if len(current_chunk) + len(content) > chunk_size and current_chunk:
                chunks.append({
                    'id': chunk_id,
                    'content': current_chunk.strip(),
                    'page': current_page
                })
                chunk_id += 1
                current_chunk = content + " "
                current_page = page
            else:
                current_chunk += content + " "
                if current_page is None:
                    current_page = page

        if current_chunk.strip():
            chunks.append({
                'id': chunk_id,
                'content': current_chunk.strip(),
                'page': current_page
            })

        return chunks

    # =========================================================================
    # Semantical chunking with Unstructured
    # =========================================================================

    @staticmethod
    def _chunk_elements_semantic(elements: List[Dict[str, Any]], max_chunk_size: int) -> List[Dict[str, Any]]:
        """
        Semantically chunk processed elements using Unstructured's text splitter.

        Args:
            elements: List of processed document elements
            max_chunk_size: Maximum characters per chunk
        
        Returns:
            List of semantically chunked elements with id, content, and page metadata
        """

        chunks = chunk_by_title(
            elements,
            max_characters=max_chunk_size,
            new_after_n_chars=int(max_chunk_size * 0.8),
            combine_text_under_n_chars = int(max_chunk_size * 0.1),
            overlap=50 # 50 characters overlap for context
        )

        # Converts into the format expected by ChromaDB
        processed_chunks = []
        for i, chunk in enumerate(chunks):
            processed_chunks.append({
                'id': i,
                'content': chunk.text,
                'page': getattr(chunk.metadata, 'page_number', None)
            })
        
        return processed_chunks

    # -------------------------------------------------------------------------
    # Indexing
    # -------------------------------------------------------------------------

    def index_document(self, pdf_path: str, processed_elements: List[Dict[str, Any]], collection_name: str = "products", chunk_size: int = CHUNK_SIZE) -> bool:
        """
        Index a PDF document with hash-based caching.
        
        This method orchestrates the indexing pipeline:
        1. Checks if document is already indexed (via SHA256 hash)
        2. If not, chunks the preprocessed content and creates the vector index
        
        Args:
            pdf_path: Path to the original PDF file (used for hash verification)
            processed_elements: List of preprocessed elements from preprocess_pdf()
            collection_name: Name for the ChromaDB collection
            chunk_size: Maximum characters per chunk
            
        Returns:
            True if indexing was performed, False if skipped (already indexed)
        """
        # Check if already indexed with same hash
        if self._is_document_indexed(pdf_path, collection_name):
            print(f"✓ Document already indexed (hash match). Loading existing index...")
            self.collection = self.client.get_collection(name=collection_name)
            return False
        
        # Document changed or not indexed - need to process
        print(f"Document not indexed or hash mismatch. Creating index...")
        
        # Delete existing collection if hash mismatch
        existing_hash = self._get_collection_hash(collection_name)
        if existing_hash is not None:
            print(f"Hash mismatch detected. Deleting old index...")
            self.client.delete_collection(name=collection_name)
        
        # Chunk the processed content
        # New implementation: semantic chunking with unstructured
        chunks = self._chunk_elements_semantic(processed_elements, chunk_size)
        # (!) chunks = self._chunk_elements(processed_elements, chunk_size)
        print(f"Created {len(chunks)} chunks")
        
        # Create index with document hash in metadata
        self._create_index(chunks, pdf_path, collection_name)
        
        return True

    def _create_index(self, chunks: List[Dict[str, Any]], source_file: str, collection_name: str = "products"):
        """
        Create vector index from chunks, storing source file hash in metadata.
        
        Args:
            chunks: List of document chunks with id, content, and metadata
            source_file: Path to source file (for hash computation)
            collection_name: Name for the ChromaDB collection
        """
        print(f"Creating vector index with {len(chunks)} chunks...")
        
        file_hash = compute_file_hash(source_file)  # Uses standalone function

        # Create new collection with cosine similarity and source hash
        self.collection = self.client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine", "source_file_hash": file_hash}
        )

        # Extract texts and generate embeddings
        texts = [chunk['content'] for chunk in chunks]
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)

        # Add to ChromaDB
        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=texts,
            ids=[str(chunk['id']) for chunk in chunks],
            metadatas=[{'page': chunk.get('page', 'unknown')} for chunk in chunks]
        )

        print(f"✓ Vector index created (hash: {file_hash[:16]}...)")

    def retrieve(self, query: str, top_k: int = TOP_K_DOCUMENTS) -> List[Dict[str, Any]]:
        """
        Retrieve most relevant documents for a query.
        
        Args:
            query: The search query (usually a compliance requirement)
            top_k: Number of documents to retrieve
            
        Returns:
            List of retrieved documents with content, page, and distance metadata
        """
        if self.collection is None:
            raise ValueError("Index not created. Call index_document() first.")

        # Generate query embedding
        query_embedding = self.embedding_model.encode([query])

        # Query ChromaDB
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )

        # Format results
        retrieved_docs = []
        for i in range(len(results['ids'][0])):
            retrieved_docs.append({
                'id': results['ids'][0][i],
                'content': results['documents'][0][i],
                'page': results['metadatas'][0][i].get('page', 'unknown'),
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
        
        return retrieved_docs

Initialize RAG and test retrieval:

In [ ]:
rag_system = RAGSystem()
was_indexed = rag_system.index_document(PRODUCT_DESCRIPTION_PDF, raw_elements, chunk_size=CHUNK_SIZE)
#  (!) OLD IMPLEMENTATION
# was_indexed = rag_system.index_document(PRODUCT_DESCRIPTION_PDF, processed_elements, chunk_size=CHUNK_SIZE)

if was_indexed:
    print("\n✓ Full indexing completed")
else:
    print("\n✓ Using cached index")

# Test retrieval
retrieved = rag_system.retrieve("security requirements", top_k=10)
for doc in retrieved:
    print(f"Page {doc['page']}: {doc['content'][:100]}...")

In [ ]:
retrieve_test = rag_system.retrieve("network scheme", top_k=1)
print(retrieve_test)

---

## 6. Decision Tree Loading

With our RAG system in place, we now need to load the decision trees that encode our compliance requirements. Each decision tree is stored as a PlantUML text file in the `DECISION_TREES_DIR` directory.

The `load_all_decision_trees` function:
1. Scans the directory for `.txt` files
2. Parses each file using our `UMLParser`
3. Returns a dictionary mapping tree names to parsed tree objects

In [ ]:
def load_all_decision_trees(trees_dir: Path) -> Dict[str, Any]:
    """
    Load all decision trees from the specified directory.
    
    Args:
        trees_dir: Path to directory containing PlantUML decision tree files
        
    Returns:
        Dictionary mapping tree names (filenames without extension) to parsed tree objects
    """
    parser = UMLParser()
    trees = {}

    tree_files = list(trees_dir.glob("*.txt"))
    print(f"Found {len(tree_files)} decision tree files")

    for tree_file in tree_files:
        try:
            tree = parser.parse_file(str(tree_file))
            tree_name = tree_file.stem
            trees[tree_name] = tree
            print(f"  ✓ Loaded tree: {tree_name}")
        except Exception as e:
            print(f"  ✗ Error loading {tree_file.name}: {e}")

    return trees

Load decision trees:

In [ ]:
trees = load_all_decision_trees(DECISION_TREES_DIR)
print(f"Loaded {len(trees)} trees: {list(trees.keys())}")

---

## 7. LLM-based Requirement Evaluation

This is the core of our compliance checker: using a **LLM** to evaluate whether the product documentation satisfies each requirement in the decision tree.

### Chain-of-Thought Prompting

We use a **structured chain-of-thought** approach to ensure:
- **Transparency**: The reasoning process is fully visible
- **Accuracy**: Breaking down analysis reduces errors
- **Traceability**: Each conclusion can be traced to specific evidence

### Evaluation Responses

The LLM must respond with exactly one of:
- **YES**: Documentation clearly satisfies the requirement
- **NO**: Documentation explicitly fails the requirement  
- **NOT APPLICABLE**: Requirement doesn't apply to this product
- **INSUFFICIENT INFO**: Documents lack needed information

### Error Handling

- **Rate limits** (OpenRouter): Automatic retry after waiting
- **Quota exceeded**: Fatal error, stops execution
- **Other LLM errors**: Fatal error, stops execution

In [ ]:
@dataclass
class EvaluationResult:
    """Result of evaluating a single requirement.
    
    Attributes:
        response: The evaluation decision (YES/NO/NOT APPLICABLE/INSUFFICIENT INFO)
        reasoning: Chain-of-thought explanation
        confidence: Confidence score between 0 and 1
        retrieved_docs: Documents used for the evaluation
    """
    response: str
    reasoning: str
    confidence: float
    retrieved_docs: List[Dict[str, Any]]


def generate_text_response(prompt: str, system_prompt: str = "", max_tokens: int = MAX_NEW_TOKENS) -> str:
    """
    Generate a text response using the configured LLM client.
    
    Args:
        prompt: The user prompt
        system_prompt: The system instruction
        max_tokens: Maximum tokens to generate
        
    Returns:
        Generated text response
        
    Raises:
        LLMError: If generation fails (fatal error)
        QuotaExceededError: If API quota is exceeded (fatal error)
    """
    return llm_client.generate(prompt, system_prompt, max_tokens)


def evaluate_requirement(
    requirement: str,
    rag_system: RAGSystem
) -> EvaluationResult:
    """
    Evaluate a single requirement using RAG + LLM.

    This function:
    1. Retrieves relevant documents using the RAG system
    2. Constructs a structured prompt with chain-of-thought instructions
    3. Calls the LLM to evaluate compliance
    4. Parses the structured response

    Args:
        requirement: The requirement question to evaluate
        rag_system: Initialized RAG system with indexed documents

    Returns:
        EvaluationResult with response, reasoning, confidence, and retrieved docs
        
    Raises:
        LLMError: If LLM request fails (fatal error)
        QuotaExceededError: If API quota is exceeded (fatal error)
    """
    # Retrieve relevant documents
    retrieved_docs = rag_system.retrieve(requirement, top_k=TOP_K_DOCUMENTS)

    # Build context from retrieved documents
    context = "\n\n".join([
        f"[Document {i+1}, Page {doc['page']}]:\n{doc['content']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    # Create prompt with explicit chain-of-thought structure
    prompt = f"""REQUIREMENT TO EVALUATE:
{requirement}

RELEVANT DOCUMENT EXCERPTS:
{context}

INSTRUCTIONS:
You are evaluating compliance. Let's think through this step by step using a structured chain of thought.

Your response MUST follow this exact structure:

CHAIN OF THOUGHT:

Step 1 - Requirement Analysis:
[What EXACTLY does this requirement ask for? Break it down into specific, verifiable criteria.]

Step 2 - Evidence Gathering:
[What information from the documents is relevant? Quote specific passages and cite document numbers, e.g., "Document 2 states: '...'"]

Step 3 - Gap Analysis:
[Compare the requirement criteria (Step 1) against the evidence (Step 2). What matches? What's missing? What contradicts?]

Step 4 - Preliminary Conclusion:
[Based on the analysis, what seems to be the answer?]

Step 5 - Verification:
[Challenge your conclusion. What assumptions did you make? Are there alternative interpretations? Is the evidence sufficient and unambiguous?]

RESPONSE: [Provide EXACTLY ONE of: YES | NO | NOT APPLICABLE | INSUFFICIENT INFO]

CONFIDENCE: [A decimal between 0.0 and 1.0]

DEFINITIONS:
- YES: The documentation clearly and explicitly satisfies the requirement
- NO: The documentation explicitly contradicts or fails the requirement
- NOT APPLICABLE: The requirement does not apply to this product/context
- INSUFFICIENT INFO: The documents lack the information needed to determine compliance
"""

    system_prompt = "You are a meticulous technical compliance expert. Your task is to analyze product documentation against regulatory requirements using structured reasoning. Always break down your analysis into explicit steps, cite specific evidence from documents, challenge your own assumptions, and provide transparent reasoning before reaching conclusions. Accuracy and traceability are paramount."

    # Call LLM - this will raise LLMError or QuotaExceededError on failure
    content = generate_text_response(prompt, system_prompt)

    # Extract structured information
    lines = content.strip().split('\n')
    response_value = "INSUFFICIENT INFO"
    confidence = 0.5
    reasoning = ""

    # Find CHAIN OF THOUGHT section (flexible matching)
    cot_start = -1
    for i, line in enumerate(lines):
        if "CHAIN OF THOUGHT" in line.upper() or line.startswith("Step 1"):
            cot_start = i
            break
    
    # Find RESPONSE
    response_idx = -1
    for i, line in enumerate(lines):
        if line.startswith("RESPONSE:"):
            response_idx = i
            response_value = line.replace("RESPONSE:", "").strip()
            # Handle pipe separator format: YES | NO | ...
            response_value = response_value.split('|')[0].strip() if '|' in response_value else response_value
            break
    
    # Extract reasoning (from CHAIN OF THOUGHT to before RESPONSE)
    if cot_start >= 0:
        if response_idx > cot_start:
            reasoning_lines = lines[cot_start:response_idx]
        else:
            reasoning_lines = lines[cot_start:]
        reasoning = "\n".join(reasoning_lines)
        reasoning = reasoning.replace("CHAIN OF THOUGHT:", "").strip()
    
    # Find CONFIDENCE
    for line in lines:
        if line.startswith("CONFIDENCE:"):
            try:
                conf_str = line.replace("CONFIDENCE:", "").strip()
                confidence = float(conf_str)
                # Ensure confidence is in valid range
                confidence = max(0.0, min(1.0, confidence))
            except:
                confidence = 0.5
            break

    return EvaluationResult(
        response=response_value,
        reasoning=reasoning,
        confidence=confidence,
        retrieved_docs=retrieved_docs
    )

Test text generation:

In [ ]:
# Test text generation
response = generate_text_response("What is compliance checking?", "Answer briefly in 2-3 sentences.")
print(f"Response: {response[:300]}...")

Test requirement evaluation:

In [ ]:
if trees:
    tree = list(trees.values())[0]
    decision_nodes = [n for n in tree._index.values() if n.kind == 'decision']
    if decision_nodes:
        result = evaluate_requirement(decision_nodes[0].text, rag_system)
        print(f"Requirement: {decision_nodes[0].text}")
        print(f"Response: {result.response}")
        print(f"Confidence: {result.confidence}")
        print(f"Reasoning: {result.reasoning}")

---

## 8. Decision Tree Navigation

Now we combine everything: the RAG system provides context, the LLM evaluates requirements, and we use these evaluations to navigate through the decision tree.

### Navigation Logic

Starting from the root node, we:
1. Move to the next node following edges from start/action/join nodes
2. For **decision/switch nodes**, evaluate the requirement using the LLM
3. Based on the response:
   - **YES**: Follow the "Yes" branch
   - **NO**: Follow the "No" branch
   - **NOT APPLICABLE/INSUFFICIENT INFO**: Stop navigation with that as the result
4. Continue until reaching a **verdict** or **terminal** node

### The TreeEvaluationResult

We track the complete evaluation including:
- Tree name and final status (COMPLETED or STOPPED)
- Final verdict (PASS/FAIL/NOT APPLICABLE, or the stopping reason)
- Complete path taken through the tree
- Number of nodes evaluated

In [ ]:
@dataclass
class TreeEvaluationResult:
    """Result of evaluating a complete decision tree.
    
    Attributes:
        tree_name: Name of the decision tree
        status: COMPLETED (reached verdict) or STOPPED (early termination)
        final_verdict: PASS/FAIL/NOT APPLICABLE, or stopping reason
        path_taken: List of nodes visited with their evaluations
        num_nodes_evaluated: Count of decision nodes evaluated
    """
    tree_name: str
    status: str
    final_verdict: Optional[str]
    path_taken: List[Dict[str, Any]]
    num_nodes_evaluated: int


def navigate_decision_tree(
    tree: Any,
    tree_name: str,
    rag_system: RAGSystem
) -> TreeEvaluationResult:
    """
    Navigate a decision tree by evaluating requirements at each decision node.

    The navigation follows these rules:
    - Start at root and follow edges from start/action/join nodes
    - At decision/switch nodes, evaluate the requirement using the LLM
    - YES response: follow the "Yes" branch
    - NO response: follow the "No" branch  
    - NOT APPLICABLE/INSUFFICIENT INFO: stop with that as the verdict
    - Stop when reaching a verdict or terminal node

    Args:
        tree: Parsed UMLDecisionTree object
        tree_name: Name identifier for the tree
        rag_system: Initialized RAG system for document retrieval

    Returns:
        TreeEvaluationResult with complete evaluation details
    """
    print(f"\n{'='*80}")
    print(f"Evaluating Decision Tree: {tree_name}")
    print(f"{'='*80}")

    path_taken = []
    current_node = tree.start
    num_nodes_evaluated = 0

    while current_node:
        node_info = {
            'node_id': current_node.id,
            'node_type': current_node.kind,
            'node_text': current_node.text,
            'evaluation': None
        }

        print(f"\n--- Node {current_node.id} ({current_node.kind}) ---")
        print(f"Text: {current_node.text[:200]}..." if len(current_node.text) > 200 else f"Text: {current_node.text}")

        # Check if we've reached a verdict
        if current_node.kind == "verdict":
            verdict = current_node.text
            print(f"\n✓ Reached verdict: {verdict}")
            node_info['verdict'] = verdict
            path_taken.append(node_info)

            return TreeEvaluationResult(
                tree_name=tree_name,
                status="COMPLETED",
                final_verdict=verdict,
                path_taken=path_taken,
                num_nodes_evaluated=num_nodes_evaluated
            )

        # Check if terminal node
        if current_node.kind == "terminal" or not current_node.edges:
            print("\n✗ Reached terminal node (detach)")
            path_taken.append(node_info)

            return TreeEvaluationResult(
                tree_name=tree_name,
                status="STOPPED",
                final_verdict=None,
                path_taken=path_taken,
                num_nodes_evaluated=num_nodes_evaluated
            )

        # For decision/switch nodes, evaluate requirement
        if current_node.kind in ["decision", "switch"]:
            requirement = current_node.text

            print(f"\nEvaluating requirement...")
            evaluation = evaluate_requirement(requirement, rag_system)
            num_nodes_evaluated += 1

            node_info['evaluation'] = {
                'response': evaluation.response,
                'reasoning': evaluation.reasoning,
                'confidence': evaluation.confidence
            }

            print(f"Response: {evaluation.response}")
            print(f"Confidence: {evaluation.confidence:.2f}")
            print(f"Reasoning: {evaluation.reasoning[:200]}..." if len(evaluation.reasoning) > 200 else f"Reasoning: {evaluation.reasoning}")

            # Decide next step based on response
            next_node = None
            
            if evaluation.response == "YES":
                # For YES response, follow the edge labeled "Yes"
                for edge in current_node.edges:
                    if edge.label and "yes" in edge.label.lower():
                        next_node = edge.target
                        break
                
                # If no "Yes" labeled edge, try first unlabeled (for old format compatibility)
                if not next_node:
                    for edge in current_node.edges:
                        if edge.label is None:
                            next_node = edge.target
                            break

                if next_node:
                    print(f"→ Following 'Yes' branch to node {next_node.id}")
                else:
                    print("✗ No 'Yes' branch found, stopping")
                    
            elif evaluation.response == "NO":
                # For NO response, follow edge labeled "No"
                for edge in current_node.edges:
                    if edge.label and "no" in edge.label.lower():
                        next_node = edge.target
                        break
                
                # If no "No" label, take the first unlabeled edge that is NOT a join node
                if not next_node:
                    for edge in current_node.edges:
                        if edge.label is None and edge.target.kind != "join":
                            next_node = edge.target
                            break
                
                if next_node:
                    print(f"→ Following 'No' branch to node {next_node.id}")
                else:
                    print("✗ No 'No' branch found, stopping")
                    
            else:
                # NOT APPLICABLE or INSUFFICIENT INFO - stop this tree
                print(f"\n✗ Stopping tree due to response: {evaluation.response}")
                path_taken.append(node_info)

                return TreeEvaluationResult(
                    tree_name=tree_name,
                    status="STOPPED",
                    final_verdict=evaluation.response,
                    path_taken=path_taken,
                    num_nodes_evaluated=num_nodes_evaluated
                )
            
            # Continue navigation if we found a next node
            if next_node:
                path_taken.append(node_info)
                current_node = next_node
            else:
                print("✗ Could not find appropriate branch, stopping")
                path_taken.append(node_info)
                break

        # For action/join nodes, just follow the edge
        elif current_node.kind in ["action", "join", "start"]:
            if current_node.edges:
                next_node = current_node.edges[0].target
                print(f"→ Following edge to node {next_node.id}")
                path_taken.append(node_info)
                current_node = next_node
            else:
                print("✗ No edges from this node, stopping")
                path_taken.append(node_info)
                break
        else:
            print(f"⚠ Unknown node type: {current_node.kind}, stopping")
            path_taken.append(node_info)
            break

    # If we exit the loop without reaching a verdict
    return TreeEvaluationResult(
        tree_name=tree_name,
        status="STOPPED",
        final_verdict=None,
        path_taken=path_taken,
        num_nodes_evaluated=num_nodes_evaluated
    )

---

## 9. Complete Compliance Check Execution

Now we execute the compliance check using all the components defined above. At this point in the notebook, we have:

- `processed_elements`: PDF content extracted with Vision LLM
- `rag_system`: Initialized RAG system with indexed document
- `trees`: All loaded decision trees

The following cells will:
1. **Evaluate each tree**: Navigate decision trees with LLM evaluation
2. **Save results**: Export to JSON and summary files
3. **Display results**: Show compliance verdicts

### Step 1: Evaluate All Decision Trees

Navigate each decision tree using the RAG system and LLM evaluation:

In [ ]:
# Evaluate each decision tree against the indexed document
print("=" * 80)
print("EVALUATING DECISION TREES")
print("=" * 80 + "\n")

results = {}

for tree_name, tree in trees.items():
    print(f"\n{'─' * 40}")
    print(f"Evaluating: {tree_name}")
    print(f"{'─' * 40}")
    
    # Navigate the tree - LLMError and QuotaExceededError will propagate up
    result = navigate_decision_tree(tree, tree_name, rag_system)
    results[tree_name] = result
    
    # Show immediate result
    print(f"  → Status: {result.status}")
    print(f"  → Verdict: {result.final_verdict or 'N/A'}")
    print(f"  → Nodes evaluated: {result.num_nodes_evaluated}")

print(f"\n{'=' * 80}")
print(f"Completed evaluation of {len(results)} decision trees")
print("=" * 80)

### Step 2: Save Results

Save evaluation results to JSON and summary text files:

In [ ]:
# Save results to JSON file
output_file = OUTPUT_DIR / "compliance_check_results.json"

serializable_results = {}
for tree_name, result in results.items():
    serializable_results[tree_name] = {
        'status': result.status,
        'final_verdict': result.final_verdict,
        'num_nodes_evaluated': result.num_nodes_evaluated,
        'path_taken': result.path_taken
    }

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(serializable_results, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to: {output_file}")

# Save human-readable summary
summary_file = OUTPUT_DIR / "compliance_summary.txt"
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("COMPLIANCE CHECK SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Document: {PRODUCT_DESCRIPTION_PDF}\n")
    f.write(f"Trees evaluated: {len(results)}\n\n")

    for tree_name, result in results.items():
        f.write(f"Tree: {tree_name}\n")
        f.write(f"  Status: {result.status}\n")
        f.write(f"  Final Verdict: {result.final_verdict or 'N/A'}\n")
        f.write(f"  Nodes Evaluated: {result.num_nodes_evaluated}\n")
        f.write("-" * 80 + "\n\n")

print(f"✓ Summary saved to: {summary_file}")

---

## 10. View Results

Let's examine the results of our compliance check. We'll display:
- A summary of all evaluated trees
- Statistics on completed vs. stopped evaluations
- Individual verdicts for each decision tree

In [ ]:
# Display summary of results
print("\n" + "=" * 80)
print("FINAL RESULTS SUMMARY")
print("=" * 80 + "\n")

completed = []
stopped = []

for tree_name, result in results.items():
    print(f"\n{tree_name}:")
    print(f"  Status: {result.status}")
    print(f"  Final Verdict: {result.final_verdict or 'N/A (stopped early)'}")
    print(f"  Nodes Evaluated: {result.num_nodes_evaluated}")

    if result.status == "COMPLETED":
        completed.append(tree_name)
    else:
        stopped.append(tree_name)

print("\n" + "=" * 80)
print(f"\n✓ Completed Trees ({len(completed)}): {', '.join(completed) if completed else 'None'}")
print(f"✗ Stopped Trees ({len(stopped)}): {', '.join(stopped) if stopped else 'None'}")

### Detailed Analysis of a Specific Tree

For deeper insights, let's examine the evaluation path of a specific decision tree. This shows:
- Each node visited during navigation
- The evaluation results for decision nodes
- The chain-of-thought reasoning
- The final verdict reached

This detailed view helps understand *why* a particular verdict was reached.

In [ ]:
# Choose a tree to analyze in detail
tree_to_analyze = list(results.keys())[4] if results else None

if tree_to_analyze:
    result = results[tree_to_analyze]

    print(f"\nDetailed Analysis: {tree_to_analyze}")
    print("=" * 80)

    for i, node in enumerate(result.path_taken, 1):
        print(f"\nStep {i}:")
        print(f"  Node ID: {node['node_id']}")
        print(f"  Type: {node['node_type']}")
        print(f"  Text: {node['node_text']}")

        if node['evaluation']:
            eval_info = node['evaluation']
            print(f"  Response: {eval_info['response']}")
            print(f"  Confidence: {eval_info['confidence']:.2f}")
            print(f"  Reasoning: {eval_info['reasoning']}")

        if 'verdict' in node:
            print(f"  🎯 VERDICT: {node['verdict']}")
else:
    print("No results available for detailed analysis.")

---

## Conclusion

In this notebook, we implemented a complete **Automated Compliance Checking System** that combines:

1. **Document Processing**: Using Unstructured and Vision LLM for comprehensive PDF parsing
2. **RAG Architecture**: Sentence transformers + ChromaDB for efficient document retrieval
3. **Decision Tree Navigation**: A custom PlantUML parser for encoding compliance logic
4. **LLM Evaluation**: Structured chain-of-thought prompting with a LLM

### Key Features

- **Automated**: Minimal human intervention required
- **Transparent**: Full reasoning trace for each decision
- **Flexible**: Easily extensible to new document types and decision trees
- **Scalable**: ChromaDB enables efficient retrieval over large document collections
- **Dual Backend**: Local llama.cpp for development, OpenRouter API for cloud execution

### LLM Backend Options

| Backend | Use Case | Model |
|---------|----------|-------|
| **llama.cpp** | Local development | Gemma 3 4B |
| **OpenRouter** | Cloud/Colab execution | `google/gemma-3-4b-it:free` |

### Error Handling

- **Rate limits**: Automatic wait and retry (OpenRouter)
- **Quota exceeded**: Fatal error with clear message
- **Connection errors**: Fatal error, stops execution

### Future Improvements

- Support for more document formats (Word, HTML, etc.)
- Interactive UI for real-time compliance checking
- Confidence calibration and uncertainty quantification
- Fine-tuning models for domain-specific compliance evaluation

---

*This notebook was developed as part of the Information Retrieval course project at the University of Pisa.*